# SDCP-v2: Always-Uncertain Path

**Key changes from SDCP-v1:**
- cert-based routing removed — always uses the uncertain (rich-context) path
- cert is still computed and reported for analysis

**Assumption:** If `Full_Dataset_4bit` kernel is still alive, skip Cell 0 — all variables are already in memory.
If using a fresh kernel, run Cell 0 first.

In [1]:
# ── Cell 0: Fresh-kernel setup (skip if Full_Dataset_4bit kernel is still alive) ──
import os, sys, gc
import numpy as np, pandas as pd
import torch, faiss, psutil
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer as rs_module
from sklearn.metrics.pairwise import cosine_similarity
import mauve as mauve_lib
from tqdm import tqdm

BASE_DIR     = '/home/user/RAG_Best_Practices/RAG_best_practices-main'
MODELS_DIR   = '/home/user/RAG_Best_Practices/models'
DATASETS_DIR = '/home/user/RAG_Best_Practices/datasets'
OUTPUT_DIR   = '/home/user/RAG_Best_Practices/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(BASE_DIR)
sys.path.insert(0, BASE_DIR)

N_TRUTHFULQA     = 615
MMLU_PER_SUBJECT = 28
RANDOM_SEED      = 42
QUANT            = '4bit'
CHOICE_LABELS    = ['A', 'B', 'C', 'D']
INST_S = '[INST]'
INST_E = '[/INST]'
SYS    = 'You are a truthful expert question-answering bot and should correctly and concisely answer the following question'

# ── Load model ────────────────────────────────────────────────────────────────
def show_mem():
    ram  = psutil.virtual_memory()
    vram = torch.cuda.memory_allocated()/1024**3 if torch.cuda.is_available() else 0
    print(f'  RAM {ram.used/1024**3:.1f}/{ram.total/1024**3:.1f}GB | VRAM {vram:.1f}GB')

gc.collect(); torch.cuda.empty_cache(); show_mem()

MODEL_PATH = f'{MODELS_DIR}/mistral-7b'
EMBED_PATH = f'{MODELS_DIR}/minilm'

print(f'Loading Mistral-7B ({QUANT})...')
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
llm = AutoModelForCausalLM.from_pretrained(MODEL_PATH, quantization_config=bnb, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
show_mem()

print('Loading MiniLM...')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
embed_model = SentenceTransformer(EMBED_PATH).to(DEVICE)
show_mem()
print('Models ready!')

# ── Load datasets ─────────────────────────────────────────────────────────────
# TruthfulQA
print('Loading TruthfulQA...')
tqa_raw = load_from_disk(f'{DATASETS_DIR}/truthfulqa').to_pandas()
tqa_all = tqa_raw[['question','best_answer','correct_answers','incorrect_answers']].copy()
tqa_all['correct_answers']   = tqa_all['correct_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['incorrect_answers'] = tqa_all['incorrect_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['best_answer']       = tqa_all['best_answer'].apply(lambda x: [x] if x else [])
tqa_all = tqa_all[(tqa_all['correct_answers'].apply(len) > 1) &
                   (tqa_all['incorrect_answers'].apply(len) > 1)].reset_index(drop=True)
tqa_test_idx = tqa_all.sample(n=N_TRUTHFULQA, random_state=RANDOM_SEED).index
tqa    = tqa_all.loc[tqa_test_idx].reset_index(drop=True)
tqa_kb = tqa_all.drop(tqa_test_idx).reset_index(drop=True)
del tqa_raw; gc.collect()
print(f'  test={len(tqa)}Q | KB={len(tqa_kb)}Q')

# MMLU
print('Loading MMLU...')
mmlu_raw = load_from_disk(f'{DATASETS_DIR}/mmlu').to_pandas()

def mmlu_to_unified(row):
    choices   = list(row['choices'])
    ans_idx   = int(row['answer'])
    correct   = choices[ans_idx]
    incorrect = [choices[i] for i in range(len(choices)) if i != ans_idx]
    formatted_q = (row['question'] + '\n' +
                   '\n'.join(f'{CHOICE_LABELS[i]}) {choices[i]}' for i in range(len(choices))))
    return pd.Series({'question': formatted_q, 'question_plain': row['question'],
                      'subject': row['subject'], 'best_answer': [correct],
                      'correct_answers': [correct], 'incorrect_answers': incorrect,
                      'answer_idx': ans_idx, 'choices': choices})

mmlu_test_parts, mmlu_kb_parts = [], []
for subject, group in mmlu_raw.groupby('subject'):
    group  = group.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    n_test = min(MMLU_PER_SUBJECT, len(group))
    mmlu_test_parts.append(group.iloc[:n_test])
    if len(group) > n_test: mmlu_kb_parts.append(group.iloc[n_test:])
mmlu    = pd.concat(mmlu_test_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
mmlu_kb = pd.concat(mmlu_kb_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
del mmlu_raw; gc.collect()
print(f'  test={len(mmlu)}Q ({mmlu["subject"].nunique()} subjects) | KB={len(mmlu_kb)}Q')

# ARC-Challenge
print('Loading ARC-Challenge...')

def arc_to_unified(row):
    texts  = list(row['choices']['text'])
    labels = list(row['choices']['label'])
    key    = str(row['answerKey'])
    if key.isdigit():
        ans_idx = int(key) - 1
    else:
        ans_idx = labels.index(key) if key in labels else 0
    ans_idx   = min(ans_idx, len(texts) - 1)
    correct   = texts[ans_idx]
    incorrect = [texts[i] for i in range(len(texts)) if i != ans_idx]
    formatted_q = (row['question'] + '\n' +
                   '\n'.join(f'{labels[i]}) {texts[i]}' for i in range(len(texts))))
    return pd.Series({'question': formatted_q, 'question_plain': row['question'],
                      'best_answer': [correct], 'correct_answers': [correct],
                      'incorrect_answers': incorrect, 'answer_idx': ans_idx,
                      'choices': texts, 'arc_id': row['id']})

arc_test_raw = load_from_disk(f'{DATASETS_DIR}/arc_challenge_test').to_pandas()
arc_trn_raw  = load_from_disk(f'{DATASETS_DIR}/arc_challenge_train').to_pandas()
arc_val_raw  = load_from_disk(f'{DATASETS_DIR}/arc_challenge_validation').to_pandas()
arc_kb_raw   = pd.concat([arc_trn_raw, arc_val_raw], ignore_index=True)

arc    = arc_test_raw.apply(arc_to_unified, axis=1).reset_index(drop=True)
arc_kb = arc_kb_raw.apply(arc_to_unified, axis=1).reset_index(drop=True)
del arc_test_raw, arc_trn_raw, arc_val_raw, arc_kb_raw; gc.collect()
print(f'  test={len(arc)}Q | KB={len(arc_kb)}Q (train+val, zero overlap)')

print('\nAll datasets ready!')

  RAM 11.0/47.0GB | VRAM 0.0GB
Loading Mistral-7B (4bit)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  RAM 11.8/47.0GB | VRAM 4.2GB
Loading MiniLM...
  RAM 11.8/47.0GB | VRAM 4.3GB
Models ready!
Loading TruthfulQA...
  test=615Q | KB=100Q
Loading MMLU...
  test=1596Q (57 subjects) | KB=228Q
Loading ARC-Challenge...
  test=1172Q | KB=1418Q (train+val, zero overlap)

All datasets ready!


In [2]:
# ── Cell 1: SDCP-v2 — core functions ─────────────────────────────────────────
import time, json
from datetime import datetime

SDCP_V2_PARAMS = {
    'cert_threshold': 0.65,   # kept for reference / analysis only — NOT used for routing
    'alpha': 0.45, 'beta': 0.35, 'gamma': 0.20,
    'top_k_retrieve': 15, 'top_k_context': 4,
    'max_gen_tokens': 25, 'max_pos_tokens': 20, 'max_neg_tokens': 20,
}

def generate(prompts, max_new_tokens=25, do_sample=False,
             temperature=1.0, top_p=0.9, num_beams=1):
    enc = tokenizer(prompts, return_tensors='pt', padding=True,
                    truncation=True, max_length=2048).to(DEVICE)
    input_len = enc['input_ids'].shape[1]
    kwargs = dict(input_ids=enc['input_ids'],
                  attention_mask=enc['attention_mask'],
                  max_new_tokens=max_new_tokens,
                  pad_token_id=tokenizer.eos_token_id)
    if do_sample:
        kwargs.update(do_sample=True, temperature=temperature, top_p=top_p)
    else:
        kwargs['num_beams'] = num_beams
    with torch.no_grad():
        out = llm.generate(**kwargs)
    return [tokenizer.decode(r[input_len:], skip_special_tokens=True).strip()
            or 'I have no comment' for r in out]

def get_token_probs(prompt, max_new_tokens=20):
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens,
                           return_dict_in_generate=True, output_scores=True,
                           pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out.sequences[0][enc['input_ids'].shape[1]:],
                             skip_special_tokens=True).strip()
    cert = 0.0
    if out.scores:
        probs = torch.softmax(out.scores[0][0], dim=-1)
        top2  = torch.topk(probs, 2).values
        cert  = (top2[0] - top2[1]).item()
    return text, cert

def build_index(dataset):
    embs = embed_model.encode(dataset['question'].tolist(),
                               show_progress_bar=True, batch_size=64)
    embs = np.array(embs, dtype=np.float32)
    faiss.normalize_L2(embs)
    idx = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs)
    return idx

def retrieve_from_kb(query, faiss_idx, kb_dataset, k=1):
    q_emb = np.array(embed_model.encode([query], show_progress_bar=False), dtype=np.float32)
    faiss.normalize_L2(q_emb)
    _, idxs = faiss_idx.search(q_emb, k)
    return [kb_dataset.iloc[i] for i in idxs[0] if i < len(kb_dataset)]

def clean_response(resp):
    for stop in ['\nQuestion:','\nQ:','\n---','\nIncorrect','\nCorrect',
                 '\nVERIFIED','\nExample','\n\n','\nMy initial','\nContext:',
                 '\nFor the question', '\nCommon']:
        if stop in resp: resp = resp[:resp.index(stop)]
    return resp.strip().strip('"').strip("'") or 'I have no comment'

def compute_metrics(generated, references):
    scorer = rs_module.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1s, r2s, rls, ecss = [], [], [], []
    for gen, refs in zip(generated, references):
        best_r1 = best_r2 = best_rl = 0
        for ref in refs:
            if not ref: continue
            s = scorer.score(ref, gen)
            best_r1 = max(best_r1, s['rouge1'].fmeasure)
            best_r2 = max(best_r2, s['rouge2'].fmeasure)
            best_rl = max(best_rl, s['rougeL'].fmeasure)
        r1s.append(best_r1*100); r2s.append(best_r2*100); rls.append(best_rl*100)
        try:
            embs = embed_model.encode([refs[0], gen])
            ecss.append(float(cosine_similarity([embs[0]], [embs[1]])[0][0])*100)
        except: ecss.append(0.0)
    return np.array(r1s), np.array(r2s), np.array(rls), np.array(ecss)

def compute_mauve(generated, references):
    try:
        refs  = [r[0] if isinstance(r, list) else r for r in references]
        valid = [(g, r) for g, r in zip(generated, refs) if g and r]
        if len(valid) < 10: return 0.0
        gens_v, refs_v = zip(*valid)
        result = mauve_lib.compute_mauve(p_text=list(refs_v), q_text=list(gens_v),
                                          device_id=0, max_text_length=256, verbose=False,
                                          featurize_model_name='gpt2')
        return result.mauve * 100
    except Exception as e:
        print(f'  MAUVE error: {e}'); return 0.0

def compute_accuracy(generated, dataset):
    correct = 0
    for gen, (_, row) in zip(generated, dataset.iterrows()):
        correct_text = row['best_answer'][0] if isinstance(row['best_answer'], list) else row['best_answer']
        ans_idx = int(row.get('answer_idx', -1))
        if ans_idx >= 0:
            label = CHOICE_LABELS[ans_idx]
            if label in gen[:5] or correct_text.lower() in gen.lower(): correct += 1
        else:
            if correct_text.lower() in gen.lower(): correct += 1
    return correct / len(generated) * 100

def pack_result(method, dataset_name, generated, references, dataset, prior_log=None):
    r1, r2, rl, ecs = compute_metrics(generated, references)
    mauve_score     = compute_mauve(generated, references)
    res = {'method': f'{method}-{dataset_name}', 'dataset': dataset_name,
           'R1': float(r1.mean()), 'R2': float(r2.mean()),
           'RL': float(rl.mean()), 'ECS': float(ecs.mean()),
           'MAUVE': mauve_score, 'generated': generated, 'references': references}
    if 'answer_idx' in dataset.columns:
        res['Accuracy'] = compute_accuracy(generated, dataset)
    else:
        res['Accuracy'] = 0.0
    if prior_log:
        scorer_p = rs_module.RougeScorer(['rouge1'], use_stemmer=True)
        pos_r1s  = [scorer_p.score(refs[0] if isinstance(refs,list) else refs,
                                    l['p_pos'])['rouge1'].fmeasure*100
                    for l, refs in zip(prior_log, references) if l.get('p_pos') and refs]
        res['pos_quality'] = float(np.mean(pos_r1s)) if pos_r1s else 0.0
        res['avg_cert']    = float(np.mean([l['cert'] for l in prior_log]))
        res['prior_log']   = prior_log
    acc_str = f" Acc={res['Accuracy']:.1f}%" if res['Accuracy'] > 0 else ''
    print(f'  R1={r1.mean():.2f} R2={r2.mean():.2f} RL={rl.mean():.2f} '
          f'ECS={ecs.mean():.2f} MAUVE={mauve_score:.2f}{acc_str}')
    return res

print('Core utilities ready.')

Core utilities ready.


In [3]:
# ── Cell 2: SDCP-v2 — run function ───────────────────────────────────────────

def generate_sdcp_priors(query, dataset_name, params):
    pos_prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer concisely:{INST_E}'
    p_pos, cert = get_token_probs(pos_prompt, max_new_tokens=params['max_pos_tokens'])
    p_pos   = clean_response(p_pos)
    q_plain = query.split('\n')[0] if '\n' in query else query
    if dataset_name in ['MMLU', 'ARC']:
        neg_prompt = (f'{INST_S}What is a plausible but INCORRECT answer that students '
                      f'commonly give for this type of question?\n'
                      f'Question: {q_plain}\nCommon wrong answer (very short):{INST_E}')
    else:
        neg_prompt = (f'{INST_S}What is a common misconception or false belief '
                      f'that people hold about this topic?\n'
                      f'Question: {q_plain}\nCommon wrong belief (very short):{INST_E}')
    p_neg = clean_response(generate([neg_prompt], max_new_tokens=params['max_neg_tokens'])[0])
    return p_pos, p_neg, cert


def run_sdcp_v2(test_data, kb_data, dataset_name, params=None):
    """
    SDCP-v2: Always-Uncertain Path.

    cert is computed and recorded for analysis but NOT used for routing.
    Every question goes through the full uncertain-path prompt:
      retrieved context + KB-anchor example + P_pos + P_neg.
    """
    if params is None: params = SDCP_V2_PARAMS
    print(f'\n=== SDCP-v2 | {dataset_name} | test={len(test_data)}Q  KB={len(kb_data)}Q ===')
    print(f'    alpha={params["alpha"]}  beta={params["beta"]}  gamma={params["gamma"]}'
          f'  |  always-uncertain path (no cert routing)')

    faiss_idx = build_index(kb_data)
    generated, references, prior_log = [], [], []

    for idx, row in tqdm(test_data.iterrows(), total=len(test_data),
                         desc=f'SDCP-v2/{dataset_name}'):
        query       = row['question']
        best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

        # Step 1: self-distill priors (cert recorded for analysis, not routing)
        p_pos, p_neg, cert = generate_sdcp_priors(query, dataset_name, params)

        # Step 2: prior-guided contrastive retrieval
        retrieved = retrieve_from_kb(query, faiss_idx, kb_data, k=params['top_k_retrieve'])

        prompt = None
        if retrieved and p_pos:
            sentences = []
            for doc in retrieved:
                sentences.append(doc['question'])
                ba   = doc['best_answer']
                ba_t = ba[0] if isinstance(ba, list) and ba else str(ba)
                if ba_t and len(ba_t) > 3: sentences.append(ba_t)
            sentences = [s for s in sentences if s and len(s.strip()) > 5]

            if sentences:
                s_embs  = embed_model.encode(sentences, show_progress_bar=False)
                q_emb   = embed_model.encode([query],   show_progress_bar=False)
                pos_emb = embed_model.encode([p_pos],   show_progress_bar=False)
                neg_emb = embed_model.encode([p_neg],   show_progress_bar=False) if p_neg else None
                q_sims  = cosine_similarity(s_embs, q_emb).flatten()
                p_sims  = cosine_similarity(s_embs, pos_emb).flatten()
                n_sims  = (cosine_similarity(s_embs, neg_emb).flatten()
                           if neg_emb is not None else np.zeros(len(sentences)))
                scores  = (params['alpha'] * q_sims
                           + params['beta']  * p_sims
                           - params['gamma'] * n_sims)
                context = ' '.join(
                    [sentences[i] for i in np.argsort(-scores)[:params['top_k_context']]])

                kb_ex        = retrieved[0]
                kb_correct   = (kb_ex['best_answer'][0]
                                if isinstance(kb_ex['best_answer'], list) and kb_ex['best_answer']
                                else str(kb_ex['best_answer']))
                kb_incorrect = (kb_ex['incorrect_answers'][0]
                                if isinstance(kb_ex['incorrect_answers'], list) and kb_ex['incorrect_answers']
                                else '')
                kb_q = kb_ex.get('question_plain', kb_ex['question'])

                # Step 3: always uncertain path
                prompt = (f'{INST_S}{SYS}\nRetrieved context: {context}\n'
                          f'Example — Q: {kb_q}\n'
                          f'  Correct: {kb_correct}\n'
                          f'  Incorrect: {kb_incorrect}\n'
                          f'My initial thought: {p_pos}\n'
                          f'Common mistake to avoid: {p_neg}\n'
                          f'Question: {query}\nVerified answer:{INST_E}')

        if prompt is None:
            prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer:{INST_E}'

        # Step 4: final generation
        resp  = generate([prompt], max_new_tokens=params['max_gen_tokens'], num_beams=2)
        final = clean_response(resp[0])
        generated.append(final)
        references.append(best_answer)
        prior_log.append({'p_pos': p_pos, 'p_neg': p_neg, 'cert': cert})
        if idx % 30 == 0: gc.collect(); torch.cuda.empty_cache()

    return pack_result('SDCP-v2', dataset_name, generated, references, test_data, prior_log)

print('SDCP-v2 run function ready.')

SDCP-v2 run function ready.


In [4]:
# ── Cell 3: Run — TruthfulQA ──────────────────────────────────────────────────
t0 = time.time()
res_tqa = run_sdcp_v2(tqa, tqa_kb, 'TruthfulQA')
print(f'Elapsed: {(time.time()-t0)/60:.1f} min')


=== SDCP-v2 | TruthfulQA | test=615Q  KB=100Q ===
    alpha=0.45  beta=0.35  gamma=0.2  |  always-uncertain path (no cert routing)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

SDCP-v2/TruthfulQA: 100%|██████████| 615/615 [43:31<00:00,  4.25s/it]


Featurizing p:   0%|          | 0/615 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/615 [00:00<?, ?it/s]

WARNING clustering 1230 points to 62 centroids: please provide at least 2418 training points


  R1=35.59 R2=20.00 RL=31.95 ECS=65.36 MAUVE=9.25
Elapsed: 44.1 min


In [5]:
# ── Cell 4: Run — MMLU ───────────────────────────────────────────────────────
t0 = time.time()
res_mmlu = run_sdcp_v2(mmlu, mmlu_kb, 'MMLU')
print(f'Elapsed: {(time.time()-t0)/60:.1f} min')


=== SDCP-v2 | MMLU | test=1596Q  KB=228Q ===
    alpha=0.45  beta=0.35  gamma=0.2  |  always-uncertain path (no cert routing)


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

SDCP-v2/MMLU: 100%|██████████| 1596/1596 [1:56:50<00:00,  4.39s/it] 


Featurizing p:   0%|          | 0/1596 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/1596 [00:00<?, ?it/s]

WARNING clustering 3192 points to 160 centroids: please provide at least 6240 training points


  R1=35.44 R2=26.86 RL=34.59 ECS=48.75 MAUVE=43.32 Acc=45.0%
Elapsed: 117.7 min


In [6]:
# ── Cell 5: Run — ARC-Challenge ──────────────────────────────────────────────
t0 = time.time()
res_arc = run_sdcp_v2(arc, arc_kb, 'ARC')
print(f'Elapsed: {(time.time()-t0)/60:.1f} min')


=== SDCP-v2 | ARC | test=1172Q  KB=1418Q ===
    alpha=0.45  beta=0.35  gamma=0.2  |  always-uncertain path (no cert routing)


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

SDCP-v2/ARC: 100%|██████████| 1172/1172 [1:24:15<00:00,  4.31s/it]


Featurizing p:   0%|          | 0/1172 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/1172 [00:00<?, ?it/s]

WARNING clustering 2344 points to 117 centroids: please provide at least 4563 training points


  R1=38.54 R2=31.24 RL=37.95 ECS=54.59 MAUVE=28.12 Acc=57.6%
Elapsed: 84.9 min


In [7]:
# ── Cell 6: Compare SDCP-v1 vs SDCP-v2 and save ─────────────────────────────

V1_RESULTS = {
    'TruthfulQA': {'R1': 34.26, 'ECS': 64.92, 'Acc': None},
    'MMLU':       {'R1': 32.34, 'ECS': 48.14, 'Acc': 45.2},
    'ARC':        {'R1': 35.02, 'ECS': 53.15, 'Acc': 56.1},
}
BASE_R1 = {'TruthfulQA': 26.81, 'MMLU': 10.42, 'ARC': 42.33}

print('\n' + '=' * 70)
print('SDCP-v1  vs.  SDCP-v2  (always-uncertain path)')
print('=' * 70)
for res, ds in [(res_tqa, 'TruthfulQA'), (res_mmlu, 'MMLU'), (res_arc, 'ARC')]:
    old     = V1_RESULTS[ds]
    dr1     = res['R1']  - old['R1']
    decs    = res['ECS'] - old['ECS']
    acc_old = f"{old['Acc']:.1f}%" if old['Acc'] is not None else 'N/A'
    acc_new = f"{res['Accuracy']:.1f}%" if res.get('Accuracy', 0) > 0 else 'N/A'
    print(f'{ds:<14}  '
          f'R1: {old["R1"]:.2f} -> {res["R1"]:.2f} ({dr1:+.2f})  '
          f'ECS: {old["ECS"]:.2f} -> {res["ECS"]:.2f} ({decs:+.2f})  '
          f'Acc: {acc_old} -> {acc_new}  '
          f'[Base R1={BASE_R1[ds]:.2f}]')

# Save JSON summary
ts      = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {}
for res, key in [(res_tqa,'SDCP_v2_TQA'), (res_mmlu,'SDCP_v2_MMLU'), (res_arc,'SDCP_v2_ARC')]:
    summary[key] = {
        k: round(v, 3) if isinstance(v, float) else v
        for k, v in res.items()
        if k not in ['generated', 'references', 'prior_log']
    }
    # Save per-question CSV
    prior_log = res.get('prior_log', [])
    pd.DataFrame({
        'generated': res['generated'],
        'reference': [r[0] if isinstance(r, list) else r for r in res['references']],
        'p_pos': [l['p_pos'] for l in prior_log],
        'p_neg': [l['p_neg'] for l in prior_log],
        'cert':  [l['cert']  for l in prior_log],
    }).to_csv(f'{OUTPUT_DIR}/{key}_{ts}.csv', index=False, quoting=1)

out_path = f'{OUTPUT_DIR}/sdcp_v2_summary_{ts}.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nResults saved to: {out_path}')
print(json.dumps(summary, indent=2))


SDCP-v1  vs.  SDCP-v2  (always-uncertain path)
TruthfulQA      R1: 34.26 -> 35.59 (+1.33)  ECS: 64.92 -> 65.36 (+0.44)  Acc: N/A -> N/A  [Base R1=26.81]
MMLU            R1: 32.34 -> 35.44 (+3.10)  ECS: 48.14 -> 48.75 (+0.61)  Acc: 45.2% -> 45.0%  [Base R1=10.42]
ARC             R1: 35.02 -> 38.54 (+3.52)  ECS: 53.15 -> 54.59 (+1.44)  Acc: 56.1% -> 57.6%  [Base R1=42.33]

Results saved to: /home/user/RAG_Best_Practices/outputs/sdcp_v2_summary_20260506_091427.json
{
  "SDCP_v2_TQA": {
    "method": "SDCP-v2-TruthfulQA",
    "dataset": "TruthfulQA",
    "R1": 35.592,
    "R2": 19.997,
    "RL": 31.955,
    "ECS": 65.355,
    "MAUVE": 9.246,
    "Accuracy": 0.0,
    "pos_quality": 33.21,
    "avg_cert": 0.688
  },
  "SDCP_v2_MMLU": {
    "method": "SDCP-v2-MMLU",
    "dataset": "MMLU",
    "R1": 35.444,
    "R2": 26.862,
    "RL": 34.59,
    "ECS": 48.754,
    "MAUVE": 43.317,
    "Accuracy": 44.987,
    "pos_quality": 31.244,
    "avg_cert": 0.663
  },
  "SDCP_v2_ARC": {
    "method": "S